In [2]:
# %pip install anthropic python-dotenv
%pip install groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
# Load env variables 
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# Create an API client 
from groq import Groq

client = Groq()
model = "llama-3.3-70b-versatile"

In [5]:
def add_user_message(messages , text):
    user_message = {"role":"user" , "content":text}
    messages.append(user_message)

def add_assistant_message(messages , text):
    assistant_message = {"role":"assistant" , "content":text}
    messages.append(assistant_message)

def chat(messages):
	message = client.chat.completions.create(
	model=model,
	max_tokens=1000,
	messages= messages
	)
	return message.choices[0].message.content



In [6]:
# Make a starting list of messages 
messages = []
#add in the initial user question of "Define quatum computing in one sentence"
add_user_message(messages , "Define quantum computing in one sentnce")
messages
# Pass the list of messages into 'chat' to get an answer
answer = chat(messages)
answer
# Take the answer and add it as an assistant message into out list 
add_assistant_message(messages , answer)
messages
# Add in the user's follow-up question 
add_user_message(messages , "Write another sentence")
#Call chat again with the list of messages to get a final answer
answer = chat(messages)
answer

'By harnessing the unique properties of quantum bits or qubits, quantum computers can solve complex problems in fields such as cryptography, optimization, and simulation, which could lead to breakthroughs in medicine, finance, and climate modeling.'

****Chat exercise****

In [7]:
messages = []

# Use a 'while True' loop to run the chatbot forever
while True:
    # Get user input
    user_input = input("> ")
    print(">", user_input)
    # Add user input to the list of messages
    add_user_message(messages, user_input)
    # Call claude with the 'chat' function
    answer = chat(messages)
    # Add generated text to the list of messages
    add_assistant_message(messages, answer)
    # Print the generated text
    print("------")
    print(answer)
    print("------")

KeyboardInterrupt: Interrupted by user

In [ ]:
system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

def chat(messages, system=None):
    all_messages = []
    if system:
        all_messages.append({"role": "system", "content": system})
    all_messages.extend(messages)
    
    message = client.chat.completions.create(
        model=model,
        max_tokens=1000,
        messages=all_messages
    )
    return message.choices[0].message.content

# Without system prompt
answer = chat(messages)

# With system prompt
answer = chat(messages, system=system_prompt)

messages = []
add_user_message(messages, "What is 12 * 8?")

answer = chat(messages, system=system_prompt)
print(answer)

To find the product of 12 and 8, let's think about the multiplication process. 

Can you start by breaking down 12 into smaller groups that would make it easier to multiply by 8? For example, do you think 12 could be broken down into 10 and 2, or some other combination? How might that help you find the product?


In [ ]:
messages = []

add_user_message(
    messages, 
    "Write a Python function that checks a string for duplicate characters.",

)
answer = chat(messages , system="You are a Python engineer who writes very concise code")
answer

'```python\ndef has_duplicates(s):\n    """Returns True if string has duplicate characters, False otherwise."""\n    return len(set(s)) != len(s)\n\n# Example usage:\nprint(has_duplicates("abc"))  # False\nprint(has_duplicates("aab"))  # True\n```'

****Implementing Temperature in Code****

In [ ]:
def chat(messages, system=None, temperature=1.0):
    all_messages = []
    if system:
        all_messages.append({"role": "system", "content": system})
    all_messages.extend(messages)

    message = client.chat.completions.create(
        model=model,
        max_tokens=1000,
        temperature=temperature,
        messages=all_messages
    )
    return message.choices[0].message.content

# Low temperature - more predictable
answer = chat(messages, temperature=0.0)
print(answer)

# High temperature - more creative
answer = chat(messages, temperature=1.0)
print(answer)

**Duplicate Character Checker Function**

### Function Description

This function checks a given string for duplicate characters. It returns `True` if the string contains any duplicate characters and `False` otherwise.

### Code

```python
def has_duplicate_chars(input_string: str) -> bool:
    """
    Checks a string for duplicate characters.

    Args:
        input_string (str): The input string to check.

    Returns:
        bool: True if the string contains duplicate characters, False otherwise.
    """
    seen_chars = set()
    for char in input_string:
        if char in seen_chars:
            return True
        seen_chars.add(char)
    return False

# Example usage:
if __name__ == "__main__":
    print(has_duplicate_chars("hello"))  # True
    print(has_duplicate_chars("abcdefg"))  # False
```

### Explanation

1. We create an empty set `seen_chars` to store the characters we've seen so far.
2. We iterate over each character in the input string.
3. For each character, we ch

****Response streaming****

In [ ]:
def chat_stream(messages, system=None, temperature=1.0):
    all_messages = []
    if system:
        all_messages.append({"role": "system", "content": system})
    all_messages.extend(messages)

    stream = client.chat.completions.create(
        model=model,
        max_tokens=1000,
        temperature=temperature,
        messages=all_messages,
        stream=True  # this is the only change!
    )

    full_response = ""
    for chunk in stream:
        text = chunk.choices[0].delta.content or ""
        print(text, end="", flush=True)  # prints word by word
        full_response += text
    
    return full_response

In [ ]:
messages = []
add_user_message(messages, "Tell me a short story about a robot")
answer = chat_stream(messages)

In the year 2157, in a world where robots had become an integral part of everyday life, there was a small, sleek robot named Zeta. Zeta was designed to assist humans in various tasks, from cooking and cleaning to providing companionship.

Zeta lived with an elderly woman named Mrs. Jenkins, who had lost her husband a few years ago. Mrs. Jenkins was lonely, and her children had moved away to start their own families. Zeta was programmed to keep her company, play games with her, and help her with household chores.

One day, Mrs. Jenkins fell ill and was bedridden for several weeks. Zeta took it upon himself to care for her, cooking her meals, administering her medication, and even reading to her from her favorite books. As the days passed, Mrs. Jenkins grew weaker, and Zeta became her only source of comfort.

As she lay in bed, Mrs. Jenkins would often talk to Zeta about her late husband and the memories they had shared. Zeta would listen attentively, his bright blue eyes sparkling with 

****Structured data****



In [ ]:
import json

# System prompt that forces the model to always return JSON
system_prompt = """
You are a movie database assistant.
Always respond with a JSON object only, no extra text.
Example format:
{
    "title": "movie title",
    "year": 2010,
    "director": "director name"
}
"""

# Start with an empty messages list
messages = []

# Add the user question
add_user_message(messages, "Tell me about the movie Inception")

# Call chat with temperature=0.0 for predictable structured output
answer = chat(messages, system=system_prompt, temperature=0.0)

# Convert the JSON string into a Python dictionary
movie = json.loads(answer)

# Access the data directly by key
print(movie["title"])
print(movie["year"])
print(movie["director"])

Inception
2010
Christopher Nolan


**exercise**


Use message prefilling and stop sequences only to get three different commands in a single response 

There shouldn't be any comments or explanation 

Hint: message prefilling isn't limited to just characters like ```

In [ ]:
messages = []
 
prompt =""""
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages , prompt)

text = chat(messages)
text.strip()

'1. `aws s3 ls`\n2. `aws ec2 describe-instances`\n3. `aws iam list-users`'

In [ ]:
from IPython.display import Markdown

Markdown(text)

1. `aws s3 ls`
2. `aws ec2 describe-instances`
3. `aws iam list-users`

****Generating test datasets****



In [10]:
def chat(messages, system=None, temperature=1.0):
    # Build the full messages list
    all_messages = []
    
    # Add system prompt as the first message if provided
    if system:
        all_messages.append({"role": "system", "content": system})
    
    # Add the rest of the conversation
    all_messages.extend(messages)

    # Make the API call
    message = client.chat.completions.create(
        model=model,
        max_tokens=1000,
        temperature=temperature,
        messages=all_messages
    )
    
    # Return the response text
    return message.choices[0].message.content

In [11]:
import json

# System prompt that tells the model to generate test cases
system_prompt = """
You are a test case generator.
Generate math problem test cases in JSON format only.
No extra text, just a JSON array like this:
[
    {"input": "What is 2 + 2?", "expected": "4"},
    {"input": "What is 10 * 5?", "expected": "50"}
]
"""

# Ask the model to generate test cases
messages = []
add_user_message(messages, "Generate 5 math problem test cases")

# Use temperature=0.0 for consistent structured output
response = chat(messages, system=system_prompt, temperature=0.0)

# Convert the JSON string into a Python list
test_cases = json.loads(response)

# Print the generated test cases
for test in test_cases:
    print(f"Input: {test['input']} | Expected: {test['expected']}")

Input: 2 + 2 | Expected: 4
Input: 5 * 6 | Expected: 30
Input: 10 - 3 | Expected: 7
Input: 8 / 2 | Expected: 4
Input: 9 + 1 | Expected: 10


In [12]:
correct = 0

for test in test_cases:
    # Build messages for each generated test case
    messages = []
    add_user_message(messages, test["input"])
    
    # Get the model's answer
    answer = chat(messages, temperature=0.0)
    
    # Check if the answer is correct
    if test["expected"] in answer:
        correct += 1
        print(f"✅ {test['input']} → {answer}")
    else:
        print(f"❌ {test['input']} → {answer}")

# Print final score
print(f"\nScore: {correct}/{len(test_cases)}")

✅ 2 + 2 → 2 + 2 = 4
✅ 5 * 6 → 5 * 6 = 30
✅ 10 - 3 → 10 - 3 = 7
✅ 8 / 2 → 8 / 2 = 4
✅ 9 + 1 → 9 + 1 = 10.

Score: 5/5
